In [22]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

In [23]:
df = pd.read_csv("/Users/prathyush/Documents/git_hub_projects/natural-language-processing-models/spacy_word_vectors/Fake_Real_Data.csv")
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [24]:
df['label_num'] = df['label'].apply(lambda x: 1 if x == 'Real' else 0)
df.head()

,Text,label,label_num
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0
1,U.S. conservative leader optimistic of common ...,Real,1
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,0
4,Democrats say Trump agrees to work on immigrat...,Real,1


In [25]:
df['vector'] = df['Text'].apply(lambda text : nlp(text).vector)
df.head

<bound method NDFrame.head of                                                    Text label  label_num  \
0      Top Trump Surrogate BRUTALLY Stabs Him In The...  Fake          0   
1     U.S. conservative leader optimistic of common ...  Real          1   
2     Trump proposes U.S. tax overhaul, stirs concer...  Real          1   
3      Court Forces Ohio To Allow Millions Of Illega...  Fake          0   
4     Democrats say Trump agrees to work on immigrat...  Real          1   
...                                                 ...   ...        ...   
9895   Wikileaks Admits To Screwing Up IMMENSELY Wit...  Fake          0   
9896  Trump consults Republican senators on Fed chie...  Real          1   
9897  Trump lawyers say judge lacks jurisdiction for...  Real          1   
9898   WATCH: Right-Wing Pastor Falsely Credits Trum...  Fake          0   
9899   Sean Spicer HILARIOUSLY Branded As Chickensh*...  Fake          0   

                                                 vector  

In [26]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(df['vector'],df['label_num'],test_size = 0.2)

In [51]:
import numpy as np

# np.stack() converts this list/array of arrays into a proper 2D numpy array:
x_train_2d = np.stack(x_train)
x_test_2d = np.stack(x_test)

In [52]:
# The error “input contains negative values” happens only with specific models in scikit-learn — especially 
# Multinomial Naive Bayes and Bernoulli Naive Bayes, because:
# Naive Bayes models DO NOT accept negative numbers
# They expect:
# word counts
# TF-IDF values
# probabilities
# These are always ≥ 0, so Naive Bayes will fail on spaCy vectors because spaCy embeddings always contain negative values.
#MinMaxScaler fits values between 0 and 1

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
sca_x_train_2d = scaler.fit_transform(x_train_2d)
sca_x_test_2d = scaler.transform(x_test_2d)

In [53]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

model = MultinomialNB()
model.fit(sca_x_train_2d,y_train)

y_pred = model.predict(sca_x_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.92      0.94      0.93       978
           1       0.94      0.92      0.93      1002

    accuracy                           0.93      1980
   macro avg       0.93      0.93      0.93      1980
weighted avg       0.93      0.93      0.93      1980



In [54]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(metric = "euclidean")
knn_model.fit(x_train_2d,y_train)
y_pred = knn_model.predict(x_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98       978
           1       0.97      0.99      0.98      1002

    accuracy                           0.98      1980
   macro avg       0.98      0.98      0.98      1980
weighted avg       0.98      0.98      0.98      1980



In [55]:
print(knnModel.n_neighbors)
print(knnModel.metric)

5
euclidean
